<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 5 Bonus · More Features, an Ensemble, and a Forecast Range
### Not one number per hour: a low, an expected and a high load for every hour

*Optional · about 45 minutes · Colab · the same Bradley weather and Connecticut load as Lab 5*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab5_Bonus_Features_Ensembles_Ranges.ipynb)

In Lab 5 your model makes **one number** per hour. ISO New England plans with more than that: how much power will
Connecticut probably need at 5 p.m. tomorrow, and how much **could** it need on a bad day? This notebook tries three
upgrades, each one an idea for the final project:

| Part | Time | What you do |
|---|---|---|
| **0. Setup** | 3 min | Same data and split as Lab 5 |
| **1. More features** | 10 min | 12 new features, all ones production could build. Do they beat Lab 5's 11? |
| **2. Three models, one ensemble** | 10 min | Gradient boosting, random forest, extra trees, and their average |
| **3. A forecast range** | 15 min | A **quantile regression forest**: low (10th percentile), expected (50th) and high (90th) load for every hour |
| **4. Tomorrow's range** | 5 min | The range for the next 24 hours from your own bucket, next to your Lab 5 forecast and ISO New England's |
| **5. Wrap up** | 2 min | Four short answers |

**Nothing here touches production.** `forecast-24h` keeps running the model you shipped in Lab 5. Part 5 lists what it
would take to ship these ideas.

# Part 0 · Setup (3 min)
The same scikit-learn as production, plus `quantile-forest` for Part 3. If Colab asks you to restart the session, click
it, then run the notebook again from the top.

In [ ]:
!pip install -q scikit-learn==1.6.1 quantile-forest==1.4.2

import io
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

print("scikit-learn", sklearn.__version__)
SEED = 42
DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])
print(len(df), "hours, from", df["hour_local"].iloc[0], "to", df["hour_local"].iloc[-1])

Lab 5's `make_features`, word for word (the 11 features production builds), and the same `score` helper.

In [ ]:
WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
CALENDAR = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]
FEATURES = CALENDAR + WEATHER + ["load_yesterday_mw"]


def make_features(hours_utc, weather, load):
    """Lab 5's 11 features: one row of model inputs per hour."""
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert("America/New_York")
    out = pd.DataFrame(index=hours_utc)
    out["hour"] = local.hour
    out["day_of_week"] = local.dayofweek
    out["month"] = local.month
    out["is_weekend"] = (local.dayofweek >= 5).astype(int)
    holidays = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
    out["is_holiday"] = local.tz_localize(None).normalize().isin(holidays).astype(int)
    for column in WEATHER:
        out[column] = weather[column].reindex(hours_utc).values
    yesterday = load.reindex(hours_utc - pd.Timedelta(hours=24)).values
    hour_before = load.reindex(hours_utc - pd.Timedelta(hours=25)).values
    out["load_yesterday_mw"] = np.where(np.isnan(yesterday), hour_before, yesterday)
    return out[FEATURES]


def score(name, actual, predicted):
    return {"model": name,
            "test_mae_mw": round(mean_absolute_error(actual, predicted), 1),
            "test_mape_pct": round(100 * mean_absolute_percentage_error(actual, predicted), 2)}

# Part 1 · More features (10 min)
Every new feature comes from the **OK rows** of the variable table in Lab 5 Part 3: production could build each one at
forecast time from the weather forecast, the clock, and load that has already happened.

| New feature | Idea |
|---|---|
| `wind_gust_mph`, `wind_dir_deg`, `precip_in` | The rest of the National Weather Service forecast |
| `day_of_year` | Finer than `month`: early June is not late June |
| `heating_degrees`, `cooling_degrees` | `max(65 - temp, 0)` and `max(temp - 65, 0)`: the two arms of the U-shape |
| `temp_3h_ago_f` | Buildings heat up and cool down slowly |
| `temp_avg_24h_f` | A hot day after a hot night is worse than a hot day after a cool night |
| `load_2_days_ago_mw`, `load_last_week_mw` | More of the past: the same hour 48 and 168 hours earlier |
| `load_yesterday_avg_mw`, `load_yesterday_peak_mw` | How big yesterday was overall, not just at this hour |

In [ ]:
EXTRA_WEATHER = ["wind_gust_mph", "wind_dir_deg", "precip_in"]


def make_features_plus(hours_utc, weather, load):
    """Lab 5's 11 features plus 12 more, every one buildable at forecast time."""
    out = make_features(hours_utc, weather, load)
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert("America/New_York")
    for column in EXTRA_WEATHER:
        out[column] = weather[column].reindex(hours_utc).values
    out["day_of_year"] = local.dayofyear
    out["heating_degrees"] = np.maximum(65 - out["temp_f"], 0)
    out["cooling_degrees"] = np.maximum(out["temp_f"] - 65, 0)
    out["temp_3h_ago_f"] = weather["temp_f"].reindex(hours_utc - pd.Timedelta(hours=3)).values
    out["temp_avg_24h_f"] = weather["temp_f"].rolling(24).mean().reindex(hours_utc).values
    out["load_2_days_ago_mw"] = load.reindex(hours_utc - pd.Timedelta(hours=48)).values
    out["load_last_week_mw"] = load.reindex(hours_utc - pd.Timedelta(hours=168)).values
    # yesterday = the 24 hours that ended 24 hours before this hour
    out["load_yesterday_avg_mw"] = load.rolling(24).mean().reindex(hours_utc - pd.Timedelta(hours=24)).values
    out["load_yesterday_peak_mw"] = load.rolling(24).max().reindex(hours_utc - pd.Timedelta(hours=24)).values
    return out


X_plus = make_features_plus(df.index, df, df["load_mw"]).dropna()    # the first week has no "last week": dropped
y_plus = df["load_mw"].reindex(X_plus.index)
PLUS = list(X_plus.columns)

years = X_plus.index.tz_convert("America/New_York").year
TEST_YEAR = years.max()
X_train = X_plus[years < TEST_YEAR]
X_test = X_plus[years == TEST_YEAR]
y_train = y_plus[years < TEST_YEAR]
y_test = y_plus[years == TEST_YEAR]
print(len(PLUS), "features |", len(X_train), "training hours |", len(X_test), "test hours (" + str(TEST_YEAR) + ")")

**A fair fight:** the same model (Lab 5's usual champion settings), the same hours, the same split. Only the feature list
changes.

In [ ]:
def lab5_booster():
    return HistGradientBoostingRegressor(learning_rate=0.05, max_iter=600, max_leaf_nodes=31,
                                         early_stopping=False, random_state=SEED)


rows = []
boost_11 = lab5_booster().fit(X_train[FEATURES], y_train)
rows.append(score("gradient boosting, Lab 5's 11 features", y_test, boost_11.predict(X_test[FEATURES])))
boost_plus = lab5_booster().fit(X_train[PLUS], y_train)
pred_boost = boost_plus.predict(X_test[PLUS])
rows.append(score("gradient boosting, all " + str(len(PLUS)) + " features", y_test, pred_boost))
rows.append(score("baseline: same hour yesterday", y_test, X_test["load_yesterday_mw"]))
pd.DataFrame(rows)

Which new features did the work? Permutation importance, the same way as Lab 5 Part 6c.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(boost_plus, X_test[PLUS], y_test, n_repeats=5, random_state=SEED,
                              scoring="neg_mean_absolute_error", n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=PLUS).sort_values()
colors = []
for name in importance.index:
    if name in FEATURES:
        colors.append("#9aa5b1")
    else:
        colors.append("#1f77b4")
plt.figure(figsize=(8, 6))
plt.barh(importance.index, importance.values, color=colors)
plt.xlabel("How much worse the MAE gets when shuffled (MW)")
plt.title("Blue = new features, gray = Lab 5's 11")
plt.show()

**Your turn (optional):** add one feature of your own to `make_features_plus` (an idea: `cooling_degrees` × `hour`, since
air conditioning in the afternoon counts more than at night), re-run Part 1, and see whether the MAE moves. Keep it to
things production could know at forecast time.

# Part 2 · Three models, one ensemble (10 min)
An **ensemble** averages several models. It helps when the models are about equally good **and** make different
mistakes. Here: gradient boosting (Lab 5's champion), a random forest, and extra trees (a random forest that picks its
split points at random, so it makes different mistakes). All three see the same 23 features.

In [ ]:
models = {
    "gradient boosting": lab5_booster(),
    "random forest": RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2, random_state=SEED, n_jobs=-1),
    "extra trees": ExtraTreesRegressor(n_estimators=200, max_depth=20, min_samples_leaf=2, random_state=SEED, n_jobs=-1),
}
preds = {}
rows = []
for name in models:
    started = time.time()
    models[name].fit(X_train[PLUS], y_train)
    preds[name] = models[name].predict(X_test[PLUS])
    rows.append(score(name, y_test, preds[name]))
    print(name, "done in", round(time.time() - started), "seconds")

equal = (preds["gradient boosting"] + preds["random forest"] + preds["extra trees"]) / 3
rows.append(score("ensemble: equal average of the three", y_test, equal))
weighted = 0.6 * preds["gradient boosting"] + 0.2 * preds["random forest"] + 0.2 * preds["extra trees"]
rows.append(score("ensemble: 60% boosting, 20% + 20%", y_test, weighted))
pd.DataFrame(rows).sort_values("test_mae_mw")

**Read the table before you celebrate.** If the ensemble loses to boosting alone, the two forests are dragging it down:
averaging a strong model with weaker ones only helps when their errors cancel out. Real forecasters choose the weights
on a **validation** period (never the test year), or tune each member until all of them are strong. Lab 5's rule
still applies: one great model often beats a committee.

# Part 3 · A forecast range (15 min)
A normal random forest averages the loads in each leaf. A **quantile regression forest** keeps **every** training load
in every leaf, so for any hour it can tell you the whole spread of loads that hours like this one had. From that spread
we read three numbers:
- **low** = the 10th percentile: only 1 hour in 10 should come in below it
- **expected** = the 50th percentile (the median)
- **high** = the 90th percentile: only 1 hour in 10 should come in above it

So about **8 in 10** real hours should land between low and high. That's the check that the range is honest.

In [ ]:
from quantile_forest import RandomForestQuantileRegressor

started = time.time()
qrf = RandomForestQuantileRegressor(n_estimators=200, min_samples_leaf=5, random_state=SEED, n_jobs=-1)
qrf.fit(X_train[PLUS], y_train)
QUANTILES = [0.1, 0.5, 0.9]
q = qrf.predict(X_test[PLUS], quantiles=QUANTILES)        # one column per quantile
ranges = pd.DataFrame({"low": q[:, 0], "expected": q[:, 1], "high": q[:, 2], "actual": y_test.values},
                      index=X_test.index.tz_convert("America/New_York"))
inside = (ranges["actual"] >= ranges["low"]) & (ranges["actual"] <= ranges["high"])
print("Fit + predict in", round(time.time() - started), "seconds")
print("Actual load inside the low-high range:", round(100 * inside.mean(), 1), "% of test hours (aim: about 80%)")
print("Average width of the range:", round((ranges["high"] - ranges["low"]).mean()), "MW")
print("Expected (median) MAE:", round(mean_absolute_error(ranges["actual"], ranges["expected"]), 1), "MW")

**The range on a real week:** the hottest week of the test year. The band is wider when the forecast is harder.

In [ ]:
hottest = X_test["temp_f"].idxmax().tz_convert("America/New_York")
week = ranges.loc[hottest - pd.Timedelta(days=3):hottest + pd.Timedelta(days=3)]
plt.figure(figsize=(11, 4))
plt.fill_between(week.index, week["low"], week["high"], alpha=0.25, label="low to high (10th to 90th percentile)")
plt.plot(week.index, week["expected"], label="expected (median)")
plt.plot(week.index, week["actual"], color="black", label="actual")
plt.ylabel("Load (MW)")
plt.title("A forecast range around the hottest hour of " + str(TEST_YEAR))
plt.legend()
plt.show()

**When is the forecast least certain?** The average width of the range, and how often the actual load landed inside it,
by hour of the day.

In [ ]:
ranges["width"] = ranges["high"] - ranges["low"]
ranges["inside"] = inside
by_hour = ranges.groupby(ranges.index.hour)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].bar(by_hour["width"].mean().index, by_hour["width"].mean().values)
ax[0].set_xlabel("Hour of the day (Connecticut time)")
ax[0].set_ylabel("Average range width (MW)")
ax[0].set_title("How wide is the range?")
ax[1].bar(by_hour["inside"].mean().index, 100 * by_hour["inside"].mean().values)
ax[1].axhline(80, color="gray", linestyle="--", label="aim: 80%")
ax[1].set_xlabel("Hour of the day (Connecticut time)")
ax[1].set_ylabel("Actual inside the range (%)")
ax[1].set_title("Is the range honest?")
ax[1].legend()
plt.show()

**The whole distribution for one hour.** The forest can give any percentile, not just three. Here is the spread for the
hottest hour of the test year: every percentile from 1 to 99.

In [ ]:
percentiles = np.arange(1, 100) / 100
one_hour = X_test[PLUS].loc[[X_test["temp_f"].idxmax()]]
spread = qrf.predict(one_hour, quantiles=list(percentiles))[0]
plt.figure(figsize=(8, 3.5))
plt.hist(spread, bins=30)
plt.axvline(y_test.loc[one_hour.index[0]], color="black", linestyle="--", label="what actually happened")
plt.xlabel("Load (MW)")
plt.ylabel("Percentiles")
plt.title("The forecast distribution for " + hottest.strftime("%b %d, %I %p"))
plt.legend()
plt.show()

# Part 4 · Tomorrow's range (5 min)
The range for the next 24 hours, built the way `forecast-24h` would: the newest weather forecast and the live load from
**your** energy bucket. Production can only build Lab 5's 11 features today (no week of live load yet), so this forest
uses those 11. Type your project and bucket, run, and pick your energy Gmail in the pop-up. No energy project? Set
`USE_MY_BUCKET = False` for a demo on the last day of the test year.

In [ ]:
PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"
BUCKET_NAME = "YOUR-ENERGY-PROJECT-ID"
USE_MY_BUCKET = True

In [ ]:
qrf_11 = RandomForestQuantileRegressor(n_estimators=200, min_samples_leaf=5, random_state=SEED, n_jobs=-1)
qrf_11.fit(X_train[FEATURES], y_train)

if USE_MY_BUCKET and "YOUR-ENERGY-PROJECT-ID" in PROJECT_ID + BUCKET_NAME:
    print("Type your PROJECT_ID and BUCKET_NAME above first. Demo mode for now.")
    USE_MY_BUCKET = False

mine = None
isone = None
if USE_MY_BUCKET:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

    def read_bucket_csv(key):
        blob = bucket.blob(key)
        if not blob.exists():
            return None
        return pd.read_csv(io.BytesIO(blob.download_as_bytes()))

    forecasts = read_bucket_csv("energy/datasets/bdl_weather_forecast_master.csv")
    forecasts["scraped_hour_utc"] = pd.to_datetime(forecasts["scraped_hour_utc"], utc=True)
    forecasts["target_hour_utc"] = pd.to_datetime(forecasts["target_hour_utc"], utc=True)
    forecasts = forecasts.sort_values(["target_hour_utc", "scraped_hour_utc"])
    weather = forecasts.drop_duplicates("target_hour_utc", keep="last").set_index("target_hour_utc")
    load_table = read_bucket_csv("energy/datasets/ct_load_5min_hourly_master.csv")
    load_table["hour_utc"] = pd.to_datetime(load_table["hour_utc"], utc=True)
    load = load_table.set_index("hour_utc")["load_mw_5min_avg"].sort_index()

    this_hour = pd.Timestamp.now(tz="UTC").floor("h")
    targets = pd.date_range(this_hour + pd.Timedelta(hours=1), periods=24, freq="h")
    X_next = make_features(targets, weather, load)
    X_next[WEATHER] = X_next[WEATHER].interpolate(limit_direction="both")      # the same gap filling as production
    X_next["load_yesterday_mw"] = X_next["load_yesterday_mw"].fillna(float(load.iloc[-1]))
    actual = load.tail(48)

    latest = read_bucket_csv("energy/forecasts/latest.csv")
    if latest is not None:
        latest["target_hour_utc"] = pd.to_datetime(latest["target_hour_utc"], utc=True)
        mine = latest.set_index("target_hour_utc")["predicted_load_mw"]
    isone_table = read_bucket_csv("energy/datasets/isone_ct_forecast_master.csv")
    if isone_table is not None:
        isone_table["target_hour_utc"] = pd.to_datetime(isone_table["target_hour_utc"], utc=True)
        isone_table = isone_table.sort_values(["target_hour_utc", "issued_at_utc"])
        isone = isone_table.drop_duplicates("target_hour_utc", keep="last").set_index("target_hour_utc")["isone_forecast_mw"]
    print("LIVE: the next 24 hours from your bucket")
else:
    targets = X_test.index[-24:]
    X_next = X_test.loc[targets, FEATURES]
    actual = y_test.loc[X_test.index[-72:-24]]
    print("DEMO: the last day of the test year")

q_next = qrf_11.predict(X_next[FEATURES], quantiles=QUANTILES)
tomorrow = pd.DataFrame({"low_mw": q_next[:, 0], "expected_mw": q_next[:, 1], "high_mw": q_next[:, 2]}, index=targets)

plt.figure(figsize=(11, 4))
plt.plot(actual.index.tz_convert("America/New_York"), actual.values, color="black", label="actual load (last 48 hours)")
hours_local = tomorrow.index.tz_convert("America/New_York")
plt.fill_between(hours_local, tomorrow["low_mw"], tomorrow["high_mw"], alpha=0.25, label="low to high")
plt.plot(hours_local, tomorrow["expected_mw"], marker="o", label="expected")
if mine is not None:
    mine_next = mine.reindex(tomorrow.index)
    plt.plot(hours_local, mine_next.values, linestyle=":", label="your Lab 5 model (deployed)")
if isone is not None:
    isone_next = isone.reindex(tomorrow.index)
    if isone_next.notna().any():
        plt.plot(hours_local, isone_next.values, linestyle="--", label="ISO New England's forecast")
plt.ylabel("Connecticut load (MW)")
plt.title("Tomorrow: low, expected and high load for every hour")
plt.legend()
plt.show()

table = tomorrow.round(0)
table.index = hours_local.strftime("%a %I %p")
table.index.name = "hour (Connecticut time)"
table

# Part 5 · Wrap up (2 min)

## Four short answers (1 or 2 sentences each)
1. How many MW did the 12 new features save over Lab 5's 11? Which new feature mattered most?
2. Did the ensemble beat gradient boosting alone? Why or why not?
3. What share of test hours landed inside the low-high range? Is the range too narrow, too wide, or about right?
4. At which hours of the day is the range widest, and why might that be?

*Your answers:*

---

## Ideas for the final project
- **Ship the better features:** copy `make_features_plus` into `make_features` in your fork's
  `cloud_function/forecast-24h/main.py` and into Lab 5 Part 3, push, wait for a green **Deploy forecast-24h**, then ship
  the model trained on them. The load from last week needs your ISO-NE scraper to have run for a full week first.
- **Ship the range:** add `quantile-forest==1.4.2` to `cloud_function/forecast-24h/requirements.txt` and write
  `low_mw` and `high_mw` columns next to `predicted_load_mw`.
- **More weather:** more stations than Bradley (Hartford, New Haven, Bridgeport), more variables, or a weather model with
  more detail. Every new input must also exist in production, every hour.
- **One great model, or a tuned ensemble:** choose the weights on a validation period, never the test year.